# STAC Search with AI Agents

This notebook demonstrates how to use `agentic-stac` to search SpatioTemporal Asset Catalogs (STAC) using natural language.

## Prerequisites

- Install the package: `pip install agentic-stac`
- For this example, we use a local Ollama model (no API key needed)
- Install Ollama from https://ollama.com and pull a model: `ollama pull qwen3`

You can also use cloud providers by setting the appropriate API key:
- OpenAI: `export OPENAI_API_KEY=sk-...`
- Anthropic: `export ANTHROPIC_API_KEY=sk-ant-...`

In [ ]:
# %pip install agentic-stac

In [ ]:
from agentic_stac import STACAgent

## Create an Agent

Create a `STACAgent` with a local Ollama model and the Element 84 Earth Search catalog.

In [ ]:
agent = STACAgent(
    model="bedrock/anthropic.claude-sonnet-4-6-20250627-v1:0",
    catalog="earth-search",
    # api_base="http://localhost:11434",
)

In [ ]:
agent = STACAgent(
    model="ollama/qwen3",
    catalog="earth-search",
    api_base="http://localhost:11434",
)

## Discover Available Collections

Ask the agent what datasets are available in the catalog.

In [ ]:
result = agent.chat("What collections are available in this catalog?")
print(result)

## Search for Satellite Imagery

Search for Sentinel-2 imagery over a specific location using natural language.

In [ ]:
result = agent.chat(
    "Find Sentinel-2 images over San Francisco from January 2024 "
    "with less than 10% cloud cover. Show me up to 5 results."
)
print(result)

## Get Collection Details

Ask for detailed information about a specific collection.

In [ ]:
result = agent.chat(
    "Tell me more about the Sentinel-2 Level-2A collection. "
    "What bands are available and what is the temporal range?"
)
print(result)

## Search with Different Parameters

Search for Landsat imagery over a different region.

In [ ]:
result = agent.chat(
    "Search for Landsat imagery over the Amazon rainforest "
    "from June 2024. Limit to 3 results with low cloud cover."
)
print(result)

## Multi-turn Conversation

The agent remembers context from previous messages, so you can have follow-up conversations.

In [ ]:
# Reset conversation to start fresh
agent.reset()

In [ ]:
result = agent.chat("What Sentinel collections are available?")
print(result)

In [ ]:
result = agent.chat(
    "Now search that first Sentinel collection for images over Tokyo "
    "from March 2024."
)
print(result)

## Using Different Catalogs

You can switch catalogs by creating a new agent.

In [ ]:
# Use Microsoft Planetary Computer
pc_agent = STACAgent(
    model="ollama/qwen3",
    catalog="planetary-computer",
    api_base="http://localhost:11434",
)

result = pc_agent.chat("What Landsat collections are available?")
print(result)

## Using the Tools Directly

You can also call the underlying STAC tools directly without the AI agent.

In [ ]:
import json
from agentic_stac.tools import geocode_location, search_stac_items, list_collections

In [ ]:
# Geocode a location
location = json.loads(geocode_location("New York City"))
print(json.dumps(location, indent=2))

In [ ]:
# Search for items using the bounding box
results = json.loads(
    search_stac_items(
        catalog_url="https://earth-search.aws.element84.com/v1",
        collections=["sentinel-2-l2a"],
        bbox=location["bbox"],
        datetime_range="2024-06-01/2024-06-30",
        query={"eo:cloud_cover": {"lt": 15}},
        max_items=5,
    )
)

print(f"Found {results['count']} items:")
for item in results["items"]:
    print(
        f"  {item['id']} | {item['datetime']} | cloud: {item['properties'].get('eo:cloud_cover', 'N/A')}%"
    )

## Using Cloud LLM Providers

To use cloud providers instead of Ollama, just change the model string:

```python
# OpenAI (set OPENAI_API_KEY env var)
agent = STACAgent(model="gpt-4.1", catalog="earth-search")

# Anthropic (set ANTHROPIC_API_KEY env var)
agent = STACAgent(model="anthropic/claude-sonnet-4-6-20250627", catalog="earth-search")

# AWS Bedrock (set AWS credentials)
agent = STACAgent(model="bedrock/anthropic.claude-sonnet-4-6-20250627-v1:0", catalog="earth-search")
```